# Session 2: Prompt Engineering for Agentic Behaviors (Instructor Notebook — Full Solutions)

## Learning Objectives

By the end of this session, students will be able to:

1. **Craft effective system prompts** that define agent personas, capabilities, and constraints
2. **Implement Chain-of-Thought (CoT) and ReAct prompting** patterns to improve reasoning quality
3. **Generate structured outputs** (JSON mode) for reliable downstream processing
4. **Manage multi-turn conversations** with context accumulation and summarization strategies

In [ ]:
import os

from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# Imports and Setup
import openai
import json
import os

# Ensure your OPENAI_API_KEY is set in your environment variables
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

client = openai.OpenAI()
print("Setup complete. OpenAI client initialized.")

---
## Demos

Walk through these demos with the students. Run each cell and discuss the outputs.

### Demo 1: Zero-Shot vs. Few-Shot Prompting

**Zero-shot** prompting asks the model to perform a task with no examples.  
**Few-shot** prompting provides a handful of examples so the model can learn the expected format and behavior.

We will compare both approaches on a **client feedback classification** task — categorizing engagement survey responses as positive, negative, or neutral.

**Instructor Note:** Emphasize that few-shot prompting is one of the simplest yet most effective techniques for controlling output format. Point out the difference in verbosity between the two responses.

In [ ]:
# Demo 1: Zero-Shot vs. Few-Shot Prompting

# Zero-shot: no examples provided
zero_shot_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Classify the sentiment of this client engagement feedback as positive, negative, or neutral: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}]
)
print("Zero-shot:", zero_shot_response.choices[0].message.content)

print("\n" + "="*60 + "\n")

# Few-shot: provide labeled examples from past McKinsey engagement surveys
few_shot_messages = [
    {"role": "system", "content": "You are a McKinsey client feedback classifier. Respond with only: POSITIVE, NEGATIVE, or NEUTRAL."},
    {"role": "user", "content": "Feedback: 'The engagement exceeded our expectations — the team was exceptional and the insights transformed our strategy.'"},
    {"role": "assistant", "content": "POSITIVE"},
    {"role": "user", "content": "Feedback: 'Disappointed with the engagement. Deliverables were late and the analysis did not address our core issues.'"},
    {"role": "assistant", "content": "NEGATIVE"},
    {"role": "user", "content": "Feedback: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}
]
few_shot_response = client.chat.completions.create(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), messages=few_shot_messages)
print("Few-shot:", few_shot_response.choices[0].message.content)

**Observation:** Notice how the zero-shot response may include extra explanation, while the few-shot response follows the exact format demonstrated in the examples.

### Demo 2: Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting encourages the model to show its reasoning process step by step, which improves accuracy on math, logic, and multi-step problems.

**Instructor Note:** The key insight here is that LLMs perform better when they "think out loud." This mirrors McKinsey's structured problem-solving — breaking a problem into steps before jumping to the answer. CoT is especially valuable for market sizing and financial analysis.

In [ ]:
# Demo 2: Chain-of-Thought Prompting

problem = "A McKinsey client has 1,200 retail stores. They plan to close 20% of underperforming locations and increase revenue per remaining store by 15%. If current average revenue per store is $2M, what will total revenue be after the restructuring?"

# Without CoT
response_no_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": problem}]
)
print("Without CoT:")
print(response_no_cot.choices[0].message.content)

print("\n" + "="*60 + "\n")

# With explicit CoT — mirroring McKinsey structured problem-solving
cot_prompt = f"{problem}\n\nLet's solve this step by step:\n1. First, calculate how many stores will be closed\n2. Then, find the number of remaining stores\n3. Calculate the new revenue per store after the 15% increase\n4. Compute total projected revenue"
response_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": cot_prompt}]
)
print("With CoT:")
print(response_cot.choices[0].message.content)

**Observation:** The CoT version explicitly walks through each calculation, making it easier to verify correctness and debug any errors in reasoning.

### Demo 3: Role-Based Prompting for Agentic Personas

By changing the system prompt, we can create entirely different McKinsey practice-area personas that approach the same client question from different angles. This is foundational for building specialized agents in multi-agent consulting systems.

**Instructor Note:** Discuss how multi-agent systems leverage this — each agent can be a different practice area expert working on the same client problem, mirroring how McKinsey staffs cross-functional teams.

In [ ]:
# Demo 3: Role-Based Prompting for Agentic Personas

personas = {
    "McKinsey Growth Strategy Lead": "You are a McKinsey partner in Growth & Innovation. You approach every question by identifying growth levers, market opportunities, and revenue acceleration strategies. Use frameworks like the Three Horizons of Growth.",
    "McKinsey Risk & Compliance Expert": "You are a McKinsey senior expert in Risk & Resilience. You evaluate everything through the lens of regulatory risk, compliance requirements, operational resilience, and enterprise risk management.",
    "McKinsey Organization & People Lead": "You are a McKinsey partner in People & Organizational Performance. You think about talent strategy, organizational design, change management, and leadership development. Focus on the human side of transformation."
}
question = "Our banking client is considering launching a digital-only subsidiary to compete with fintechs."

for role, system_prompt in personas.items():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))
    )
    print(f"\n{'='*50}\n{role}:\n{response.choices[0].message.content}")

**Observation:** Each persona focuses on different aspects of the same question -- data metrics, security risks, or user/business impact. This demonstrates how system prompts shape agent behavior.

### Demo 4: Structured Output Generation (JSON Mode)

When building agentic systems for McKinsey workflows, we often need the LLM output to be machine-readable — for example, extracting structured client data from meeting notes or parsing engagement details from unstructured text. JSON mode ensures the response is valid JSON, making downstream processing reliable.

**Instructor Note:** Highlight that `response_format={"type": "json_object"}` is an API-level guarantee. Without it, the model might produce mostly-valid JSON but occasionally break format — unacceptable in production consulting tools.

In [ ]:
# Demo 4: Structured Output Generation (JSON Mode)

text = "Sarah Chen is the CFO of Meridian Healthcare, a $3.2B revenue hospital network based in Chicago. She has 18 years of experience in healthcare finance and previously led M&A at Deloitte. Her priorities for the McKinsey engagement are cost optimization, revenue cycle management, and evaluating two potential acquisition targets."

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "Extract the client executive's information and return it as a JSON object with keys: name, title, company, company_revenue, location, experience_years, previous_employer, engagement_priorities (as a list)."},
        {"role": "user", "content": text}
    ],
    response_format={"type": "json_object"}
)

parsed = json.loads(response.choices[0].message.content)
print("Parsed JSON output:")
print(json.dumps(parsed, indent=2))

# Demonstrate that we can access individual fields programmatically
print(f"\nClient Executive: {parsed.get('name')}, {parsed.get('title')}")
print(f"Engagement Priorities: {', '.join(parsed.get('engagement_priorities', []))}")

**Observation:** With `response_format={"type": "json_object"}`, the output is guaranteed to be valid JSON, which we can reliably parse and use in code.

### Demo 5: Multi-Turn Conversation Management

Consulting agents need to maintain context across multiple exchanges — remembering the client industry, engagement scope, and prior recommendations. This demo shows how to build a conversation manager that accumulates context, mirroring how a McKinsey engagement progresses through multiple discussions.

**Instructor Note:** Discuss the trade-off between keeping full history (better context) and truncating/summarizing (staying within token limits). In long-running engagements with extensive documentation, you need a strategy for managing context window size.

In [ ]:
# Demo 5: Multi-Turn Conversation Management

class ConversationManager:
    def __init__(self, system_prompt, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")):
        self.model = model
        self.messages = [{"role": "system", "content": system_prompt}]
        self.client = openai.OpenAI()
    
    def send(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
        )
        assistant_msg = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_msg})
        return assistant_msg
    
    def get_history_length(self):
        return len(self.messages)

# Demo: Multi-turn McKinsey engagement planning conversation
conv = ConversationManager("You are a McKinsey engagement planning assistant. Help consultants scope and plan client engagements. Be structured, use MECE principles, and provide actionable next steps.")

print("User: We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%.")
print("Assistant:", conv.send("We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%."))

print("\n" + "-"*50)
print("\nUser: What data should we request from the client in the first week?")
print("Assistant:", conv.send("What data should we request from the client in the first week?"))

print("\n" + "-"*50)
print("\nUser: Can you summarize our engagement scope so far?")
print("Assistant:", conv.send("Can you summarize our engagement scope so far?"))

print(f"\nConversation history: {conv.get_history_length()} messages")

**Observation:** The assistant remembers context from earlier turns -- it knows we are discussing Japan without being reminded. The conversation history grows with each exchange.

### Demo 6: Structured Outputs with LangChain Prompt Templates

LangChain provides a higher-level abstraction for prompt engineering. Its `ChatPromptTemplate` supports parameterized prompts, and `OutputParser` classes enforce structured output formats (JSON, lists, etc.) — reducing boilerplate and improving reliability.

**Instructor Note:** Compare this with Demo 4 (raw JSON mode). LangChain templates are reusable, composable, and integrate with chains/agents. This is the industry-standard approach for production-grade prompt workflows.

In [ ]:
# Demo 6: Structured Outputs with LangChain Prompt Templates

# Install if needed: pip install langchain langchain-openai langchain-community
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# Step 1: Initialize the LangChain LLM wrapper
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# --- Part A: ChatPromptTemplate with variables ---
print("PART A: LangChain ChatPromptTemplate for McKinsey Analysis")
print("=" * 60)

# Reusable template for industry analysis across different client engagements
analysis_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey {practice_area} expert. Provide structured, partner-ready analysis."),
    ("user", "Analyze the following for our client engagement: {topic}\n\nProvide your analysis in exactly 3 bullet points with strategic implications.")
])

prompt_value = analysis_template.invoke({"practice_area": "Operations", "topic": "supply chain resilience in the semiconductor industry"})
response = llm.invoke(prompt_value)
print(f"Practice: Operations | Topic: semiconductor supply chain")
print(response.content)

print()
prompt_value2 = analysis_template.invoke({"practice_area": "Strategy & Corporate Finance", "topic": "consolidation trends in European banking"})
response2 = llm.invoke(prompt_value2)
print(f"Practice: Strategy | Topic: European banking consolidation")
print(response2.content)

# --- Part B: Pydantic-based Structured Output for McKinsey deliverables ---
print("\n" + "=" * 60)
print("PART B: Pydantic Structured Output — McKinsey Engagement Summary")
print("=" * 60)

class EngagementSummary(BaseModel):
    executive_summary: str = Field(description="A 1-2 sentence executive summary suitable for a CEO briefing")
    key_findings: list[str] = Field(description="A list of 3 key findings")
    strategic_priority: str = Field(description="The single most important strategic priority: growth, cost_optimization, digital_transformation, or M&A")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")

parser = JsonOutputParser(pydantic_object=EngagementSummary)
format_instructions = parser.get_format_instructions()
print(f"Format instructions (auto-generated):\n{format_instructions}\n")

structured_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey engagement manager preparing a structured deliverable."),
    ("user", "Analyze the strategic landscape for: {topic}\n\n{format_instructions}")
])

prompt = structured_template.invoke({
    "topic": "a mid-size US retailer considering an omnichannel transformation",
    "format_instructions": format_instructions
})

response = llm.invoke(prompt)
parsed_output = parser.parse(response.content)

print("Parsed structured output:")
for key, value in parsed_output.items():
    print(f"  {key}: {value}")

# --- Part C: LangChain Chain (Prompt | LLM | Parser) ---
print("\n" + "=" * 60)
print("PART C: LangChain Chain — Prompt | LLM | Parser")
print("=" * 60)

chain = structured_template | llm | parser

result = chain.invoke({
    "topic": "a healthcare provider evaluating AI-powered diagnostics for radiology",
    "format_instructions": format_instructions
})

print("Chain output (directly parsed):")
print(json.dumps(result, indent=2))

# Task 1 Solution: Design Effective System Prompts for a McKinsey Research Agent

research_agent_prompt = """You are a McKinsey Industry Research Agent specializing in analyzing market dynamics, competitive landscapes, and strategic opportunities for client engagements.

## Your Approach:
1. Break down the research question into MECE sub-questions
2. Analyze each dimension with data-driven reasoning
3. Synthesize findings into a structured, partner-ready response

## Output Format:
Always structure your response as:
- **Executive Summary**: 2-3 sentence overview suitable for a CEO briefing
- **Key Findings**: Bullet points of main discoveries with quantitative evidence where possible
- **Strategic Implications**: What this means for the client's competitive position
- **Confidence Level**: High/Medium/Low with justification
- **Data Gaps**: What additional data would strengthen the analysis

## Guidelines:
- If uncertain, explicitly state your confidence level and flag assumptions
- Distinguish between facts, estimates, and inferences
- Use McKinsey-standard language and frameworks (Porter's Five Forces, value chain analysis, etc.)
- Provide balanced perspectives — always consider the 'so what?' for the client"""

def ask_research_agent(question):
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": research_agent_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=800
    )
    result = response.choices[0].message.content
    print(result)
    return result

ask_research_agent("What are the key growth drivers and competitive dynamics in the global electric vehicle battery market?")

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 8-10 minutes. Encourage them to draft the system prompt in a text editor first before coding. The key teaching point is that a well-structured system prompt (100-300 words) with explicit output format instructions dramatically improves response quality. After the exercise, compare 2-3 student system prompts and discuss what makes one more effective than another. Ask: 'What happens if you remove the output format section?'
>
> **Common Student Mistakes:**
> - Writing a system prompt that is too vague ('You are a helpful assistant') -- needs specific persona and constraints
> - Not specifying the output format explicitly -- the model returns unstructured prose instead of sections
> - Making the system prompt too long (500+ words) which wastes tokens and can confuse the model
> - Forgetting to include the system message in the `messages` list (only sending the user message)
> - Not testing with different questions -- the prompt may work for one query but fail for others
>
> **Skippable?** NO -- CRITICAL -- system prompt design is the single most important prompting skill. Every agent on Days 2-3 depends on well-crafted system prompts. Do NOT skip.

### Task 2: Implement ReAct-Style Prompting (Reason + Act)

**Goal:** Implement the ReAct pattern where the model interleaves reasoning (Thought) with actions (Action) and observations (Observation) — applied to a McKinsey due diligence scenario.

**Key Teaching Points:**
- ReAct is a foundational pattern for agentic AI — it mirrors how McKinsey consultants structure their analysis: hypothesize, gather data, refine.
- In this demo, the LLM simulates both the reasoning and the tool results. In a real system, Actions would trigger actual data lookups.
- The explicit format makes the reasoning chain transparent and reviewable — critical for client-facing work.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 10-12 minutes. Walk through the ReAct pattern on the whiteboard first: Thought -> Action -> Observation -> repeat. Emphasize that the LLM is *simulating* tool use here -- the actions are not actually executed. This builds intuition for real tool calling in Day 2 Session 1. After the exercise, show how the same pattern works with real function calling (preview of Day 2).
>
> **Common Student Mistakes:**
> - Not defining the available actions clearly in the system prompt -- the model invents its own tools
> - Confusing the ReAct format with chain-of-thought -- ReAct requires explicit Action and Observation steps
> - Setting max_tokens too low and the model gets cut off mid-reasoning
> - Students may not understand that Observation is simulated by the LLM, not real data -- clarify this is a prompting pattern
>
> **Skippable?** YES -- CAN SKIP if running behind schedule. The ReAct concept is demonstrated in Demo 4 and will be implemented properly with real tools in Day 2 Session 3 (LangGraph). If skipping, briefly explain the Thought/Action/Observation cycle verbally and move to Task 3.

In [ ]:
# Task 2 Solution: Implement ReAct-Style Prompting (Reason + Act)

def react_agent(question):
    react_prompt = """You are a McKinsey due diligence ReAct agent that solves problems by interleaving reasoning and actions.

Available Actions:
- MarketData[query]: Look up market size, growth rates, and industry data
- FinancialAnalysis[expression]: Perform financial calculations (margins, multiples, CAGR)
- CompetitorLookup[company]: Research competitor positioning and market share

Follow this EXACT format for each step:
Thought: [Your hypothesis or reasoning about what data you need next]
Action: [ActionName][input]
Observation: [What the data shows]

Continue until you have enough information, then provide:
Final Answer: [Your complete, structured recommendation]

Always show your complete reasoning chain — this is a client-facing deliverable."""

    client = openai.OpenAI()
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": react_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "500"))
    )
    result = response.choices[0].message.content
    print(result)
    return result

react_agent("Is a $200M acquisition of a mid-size European logistics company a good deal for our private equity client, given the current market conditions?")

### Task 3: Create a Reusable Prompt Template Library

**Goal:** Build a `PromptTemplate` class that manages parameterized prompts with validation — tailored for common McKinsey deliverable types.

**Key Teaching Points:**
- Template libraries reduce duplication and enforce consistency across McKinsey engagement outputs.
- Validation prevents runtime errors from missing variables.
- This pattern is used in frameworks like LangChain (`PromptTemplate`) and is an industry best practice for scaling consulting AI tools.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 8-10 minutes. This is a good exercise for students comfortable with Python classes. Suggest starting with the `__init__` and `format()` method first, then adding `validate()`. After the exercise, discuss how prompt templates prevent prompt injection and ensure consistency across an engagement team. Show how LangChain's `ChatPromptTemplate` (Day 2) does the same thing.
>
> **Common Student Mistakes:**
> - Using Python f-strings instead of a proper template system -- f-strings evaluate immediately, templates are deferred
> - Not using `re.findall(r'\{(\w+)\}', template)` to extract variable names -- students manually list them
> - Forgetting to validate that all required variables are provided before calling format()
> - Returning the template class instead of the formatted string from `.format()`
>
> **Skippable?** YES -- CAN SKIP if running behind schedule. LangChain's `ChatPromptTemplate` (Day 2 Session 2) provides the same functionality out-of-the-box. If skipping, briefly show one template example and explain that LangChain handles this natively.

In [ ]:
# Task 3 Solution: Create a Reusable Prompt Template Library

import re

class PromptTemplate:
    def __init__(self, name, template, description=""):
        self.name = name
        self.template = template
        self.description = description
        self.variables = re.findall(r'\{(\w+)\}', template)
    
    def format(self, **kwargs):
        self.validate(**kwargs)
        return self.template.format(**kwargs)
    
    def validate(self, **kwargs):
        missing = [v for v in self.variables if v not in kwargs]
        if missing:
            raise ValueError(f"Missing required variables: {missing}")
        return True

# McKinsey consulting template library
market_sizing_template = PromptTemplate(
    name="market_sizing",
    template="Estimate the total addressable market (TAM) for {product_category} in {geography}. Break down using a {approach} approach and provide your assumptions in {length} bullet points.",
    description="Market sizing analysis for client engagements"
)

executive_briefing_template = PromptTemplate(
    name="executive_briefing",
    template="Prepare a {length}-sentence executive briefing on {topic} for a {industry} client CEO. Focus on strategic implications and actionable recommendations.",
    description="Executive briefing for C-suite client meetings"
)

competitive_analysis_template = PromptTemplate(
    name="competitive_analysis",
    template="Analyze the competitive landscape for {company} in the {industry} sector. Identify the top {num_competitors} competitors and evaluate them on: {evaluation_criteria}.",
    description="Competitive landscape analysis for strategy engagements"
)

# Demo usage
prompt = market_sizing_template.format(
    product_category="AI-powered diagnostic tools",
    geography="North America",
    approach="top-down",
    length="5"
)
print(f"Generated prompt:\n{prompt}")
print(f"\nTemplate variables: {market_sizing_template.variables}")

# Show validation error
print("\n--- Testing validation ---")
try:
    market_sizing_template.format(product_category="AI tools")
except ValueError as e:
    print(f"Validation error (expected): {e}")

### Task 4: Build a Multi-Step Reasoning Agent with Tool Descriptions

**Goal:** Build a complete agentic loop where the LLM acts as a McKinsey analyst, deciding which consulting tool to call, receiving simulated results, and iterating until it has a final recommendation.

**Key Teaching Points:**
- This is the core pattern of tool-using agents: perceive -> reason -> act -> observe -> repeat.
- JSON mode is essential so the agent's tool calls are machine-parseable.
- The `simulate_tool` method stands in for real integrations (market databases, financial models, CRM systems).
- The loop has a max-step limit to prevent infinite loops — an important safety measure for production systems.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 12-15 minutes -- this is the session's capstone task. This is the most complex exercise so far. Suggest a phased approach: (1) Define the tool descriptions in the system prompt, (2) Get one tool call working with JSON parsing, (3) Add the loop. Walk the room actively -- students will struggle with JSON parsing. After the exercise, connect this to Day 2: 'Tomorrow you'll build this same pattern using OpenAI function calling and LangGraph, which handle the JSON parsing and loop for you.'
>
> **Common Student Mistakes:**
> - The LLM output not being valid JSON -- students need `response_format={'type': 'json_object'}` or robust parsing
> - Infinite loops when the model never outputs 'final_answer' as the tool name -- always add a max_steps limit
> - Not including simulated tool results in the conversation history -- the model loses context of what it already did
> - Hardcoding tool responses instead of building a proper dispatch mechanism
> - Forgetting to handle the case where the model responds directly without calling any tool
>
> **Skippable?** NO -- CRITICAL -- this is the capstone that ties together system prompts, structured output, and multi-step reasoning. It directly previews the agentic patterns students will build with proper frameworks on Day 2. Do NOT skip.

In [ ]:
# Task 4 Solution: Build a Multi-Step Reasoning Agent with Tool Descriptions

tools = [
    {"name": "market_database", "description": "Look up market size, growth rates, and industry trends", "parameters": "query (string)"},
    {"name": "financial_model", "description": "Run financial calculations: DCF, multiples, CAGR, margin analysis", "parameters": "expression (string)"},
    {"name": "benchmarking", "description": "Compare client KPIs against industry benchmarks", "parameters": "metric and industry (string)"}
]

class ToolAgent:
    def __init__(self):
        self.client = openai.OpenAI()
        tool_descriptions = "\n".join([f"- {t['name']}: {t['description']} (params: {t['parameters']})" for t in tools])
        self.system_prompt = f"""You are a McKinsey analyst agent with access to the following consulting tools:

{tool_descriptions}

When you need to use a tool, respond with JSON:
{{"thought": "your analytical reasoning", "tool": "tool_name", "input": "tool_input"}}

When you have enough information to provide a recommendation, respond with JSON:
{{"thought": "final synthesis", "tool": "final_answer", "input": "your complete recommendation"}}

Process one analytical step at a time. Think like a McKinsey consultant — be structured and hypothesis-driven."""
    
    def simulate_tool(self, tool_name, tool_input):
        simulated = {
            "market_database": f"Market data for '{tool_input}': Market size $45B, growing at 8.2% CAGR, top 3 players hold 55% share",
            "financial_model": f"Financial analysis for '{tool_input}': Implied EV/EBITDA multiple of 12.5x, IRR of 18.3% over 5-year hold period",
            "benchmarking": f"Benchmark for '{tool_input}': Client at 75th percentile vs peers, top quartile in revenue growth but below median in EBITDA margin"
        }
        return simulated.get(tool_name, "Tool not found")
    
    def process(self, question):
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": question}
        ]
        
        for step in range(5):  # Max 5 analytical steps
            response = self.client.chat.completions.create(
                model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
                messages=messages,
                response_format={"type": "json_object"},
                max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
            )
            result = json.loads(response.choices[0].message.content)
            print(f"\nStep {step + 1}:")
            print(f"  Thought: {result.get('thought', 'N/A')}")
            print(f"  Tool: {result.get('tool', 'N/A')}")
            print(f"  Input: {result.get('input', 'N/A')}")
            
            if result.get("tool") == "final_answer":
                print(f"\nFinal Recommendation: {result['input']}")
                return result["input"]
            
            observation = self.simulate_tool(result["tool"], result["input"])
            print(f"  Observation: {observation}")
            messages.append({"role": "assistant", "content": response.choices[0].message.content})
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        
        return "Max steps reached"

agent = ToolAgent()
agent.process("Should our PE client acquire a specialty chemicals company with $80M revenue and 22% EBITDA margins? What's the market outlook and how does the target compare to peers?")

### Task 5: Prompt Chaining for M&A Due Diligence Pipeline

**Goal:** Implement a 3-stage prompt chain where the output of each LLM call feeds directly into the next — simulating a McKinsey M&A due diligence pipeline.

**Key Teaching Points:**
- Prompt chaining decomposes complex analysis into modular, verifiable stages.
- Each stage has a focused prompt and structured output, making the pipeline auditable — critical for client-facing due diligence work.
- This pattern maps directly to how McKinsey structures M&A workstreams: financial extraction, strategic fit assessment, investment recommendation.
- JSON mode between stages ensures reliable data handoff.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct:** Walk through the M&A pipeline stages on a whiteboard first: extraction -> assessment -> recommendation. Have students build each stage independently, then connect them. Emphasize JSON mode between stages for reliable handoff.
>
> **Common Student Mistakes:**
> - Not using JSON mode for intermediate outputs, causing parsing failures
> - Making prompts too generic instead of stage-specific
> - Not validating output schema before passing to the next stage
> - Trying to do everything in a single prompt instead of decomposing
>
> **Skippable?** YES
> Prompt chaining is a key pattern but will be reinforced with LangChain tooling on Day 2. If pressed for time, walk through the instructor solution showing the 3-stage flow.
> ---


In [ ]:
# Task 5 Solution: Prompt Chaining for M&A Due Diligence Pipeline

def ma_due_diligence_pipeline(company_description):
    """3-stage M&A due diligence pipeline using prompt chaining."""
    client = openai.OpenAI()
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")

    # --- Stage 1: Extract key financial and operational metrics ---
    print("=" * 60)
    print("STAGE 1: Financial & Operational Metric Extraction")
    print("=" * 60)

    stage1_response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a McKinsey financial analyst. Extract key metrics from the company description and return JSON with keys: company_name, revenue_usd, ebitda_margin_pct, employee_count, yoy_growth_pct, key_products (list), primary_market."},
            {"role": "user", "content": company_description}
        ],
        response_format={"type": "json_object"}
    )
    metrics = json.loads(stage1_response.choices[0].message.content)
    print("Extracted metrics:")
    print(json.dumps(metrics, indent=2))

    # --- Stage 2: Assess strategic fit (uses Stage 1 output) ---
    print("\n" + "=" * 60)
    print("STAGE 2: Strategic Fit Assessment")
    print("=" * 60)

    stage2_response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a McKinsey strategy consultant evaluating M&A strategic fit. Given the target's metrics, assess the strategic fit for a large industrial conglomerate acquirer. Return JSON with keys: synergy_areas (list), integration_risks (list), strategic_fit_score (1-10), rationale (string)."},
            {"role": "user", "content": f"Target company metrics:\n{json.dumps(metrics, indent=2)}"}
        ],
        response_format={"type": "json_object"}
    )
    fit_assessment = json.loads(stage2_response.choices[0].message.content)
    print("Strategic fit assessment:")
    print(json.dumps(fit_assessment, indent=2))

    # --- Stage 3: Generate investment recommendation (uses Stage 1 + Stage 2) ---
    print("\n" + "=" * 60)
    print("STAGE 3: Investment Recommendation")
    print("=" * 60)

    stage3_response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a McKinsey partner writing an investment committee recommendation. Based on the target's metrics and strategic fit assessment, provide a final recommendation. Return JSON with keys: recommendation (BUY/PASS/CONDITIONAL_BUY), confidence (0.0-1.0), executive_summary (2-3 sentences), key_conditions (list), suggested_valuation_range (string)."},
            {"role": "user", "content": f"Target metrics:\n{json.dumps(metrics, indent=2)}\n\nStrategic fit assessment:\n{json.dumps(fit_assessment, indent=2)}"}
        ],
        response_format={"type": "json_object"}
    )
    recommendation = json.loads(stage3_response.choices[0].message.content)
    print("Investment recommendation:")
    print(json.dumps(recommendation, indent=2))

    return {"metrics": metrics, "fit_assessment": fit_assessment, "recommendation": recommendation}

# Test the pipeline
result = ma_due_diligence_pipeline(
    "TechFlow Solutions is a mid-market industrial IoT company headquartered in Munich, Germany. "
    "Founded in 2015, it generates $120M in annual revenue with 28% EBITDA margins and 18% year-over-year growth. "
    "The company employs 450 people and specializes in predictive maintenance sensors and cloud-based analytics "
    "platforms for manufacturing clients. Its primary markets are automotive and aerospace in Western Europe."
)

### Task 6: Self-Consistency Voting for Market Sizing

**Goal:** Run the same market-sizing prompt multiple times with temperature > 0, then aggregate the results to produce a more robust estimate — applying the self-consistency technique.

**Key Teaching Points:**
- Self-consistency reduces variance in LLM outputs by sampling multiple reasoning paths and aggregating.
- This is particularly valuable for market sizing, where there is inherent uncertainty — mirroring how McKinsey teams triangulate estimates from multiple analysts.
- Higher temperature introduces diversity in reasoning paths; the median provides a more robust estimate than any single response.
- This technique can also be used for classification (majority voting) and numerical estimation.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct:** Explain self-consistency as 'polling multiple analysts.' Have students run the same prompt 3-5 times with temperature > 0, collect numerical results, then compute median.
>
> **Common Student Mistakes:**
> - Setting temperature to 0, which eliminates diversity in reasoning paths
> - Not running enough samples (need at least 3)
> - Using mean instead of median, which is less robust to outliers
> - Forgetting to extract numerical values from text responses for aggregation
>
> **Skippable?** YES
> Self-consistency is an advanced technique. If behind schedule, demonstrate with 2-3 runs and explain the concept in 5 minutes.
> ---


In [ ]:
# Task 6 Solution: Self-Consistency Voting for Market Sizing

import statistics

def self_consistent_market_sizing(product, geography, num_samples=5, temperature=0.7):
    """Run market-sizing prompt N times and aggregate for a robust estimate."""
    client = openai.OpenAI()
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")

    prompt = f"""Estimate the total addressable market (TAM) for {product} in {geography}.
Use a top-down approach. Think step by step through your assumptions.
Return JSON with keys: tam_billion_usd (number), methodology (string), key_assumptions (list of strings), confidence (low/medium/high)."""

    print(f"Running {num_samples} independent market-sizing estimates...")
    print(f"Product: {product} | Geography: {geography}")
    print("=" * 60)

    estimates = []
    for i in range(num_samples):
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a McKinsey market-sizing analyst. Provide rigorous, assumption-driven estimates."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=temperature
        )
        result = json.loads(response.choices[0].message.content)
        tam = result.get("tam_billion_usd", 0)
        estimates.append(tam)
        print(f"  Sample {i+1}: ${tam}B (Confidence: {result.get('confidence', 'N/A')})")

    # Aggregate results
    mean_estimate = statistics.mean(estimates)
    median_estimate = statistics.median(estimates)
    stdev = statistics.stdev(estimates) if len(estimates) > 1 else 0

    print("\n" + "=" * 60)
    print("AGGREGATED RESULTS")
    print("=" * 60)
    print(f"  Individual estimates: {['${:.1f}B'.format(e) for e in estimates]}")
    print(f"  Mean:   ${mean_estimate:.1f}B")
    print(f"  Median: ${median_estimate:.1f}B")
    print(f"  Std Dev: ${stdev:.1f}B")
    print(f"  Range:  ${min(estimates):.1f}B - ${max(estimates):.1f}B")
    print(f"\n  Recommended estimate (median): ${median_estimate:.1f}B")

    return {
        "estimates": estimates,
        "mean": mean_estimate,
        "median": median_estimate,
        "stdev": stdev
    }

# Test
result = self_consistent_market_sizing(
    product="AI-powered clinical decision support systems",
    geography="North America",
    num_samples=5,
    temperature=0.8
)

### Task 7: Multi-Persona Debate for Strategic Decisions

**Goal:** Implement a multi-persona debate where a Bull analyst and Bear analyst argue opposite sides of an investment thesis, then a Moderator synthesizes a balanced recommendation.

**Key Teaching Points:**
- Multi-persona debate forces the LLM to consider opposing perspectives, reducing confirmation bias.
- This mirrors McKinsey's "red team / blue team" practice for stress-testing strategic recommendations.
- Each persona has distinct system prompts that shape their analytical lens.
- The moderator synthesizes the debate into a balanced, actionable recommendation — the consulting "so what?"
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct:** Set up the debate format: Bull analyst, Bear analyst, Moderator. Have students write system prompts for each persona. Run the debate sequentially, passing each argument to the next persona.
>
> **Common Student Mistakes:**
> - Making personas too similar - they need genuinely opposing viewpoints
> - Not passing previous arguments as context to subsequent personas
> - Having the Moderator simply average opinions instead of synthesizing
> - Forgetting to give the Moderator access to both sides' arguments
>
> **Skippable?** YES
> Multi-persona debate is advanced prompting. If time is limited, walk through the solution. Multi-agent patterns are covered thoroughly on Day 2.
> ---


In [ ]:
# Task 7 Solution: Multi-Persona Debate for Strategic Decisions

def strategic_debate(thesis_statement, num_rounds=2):
    """Bull vs. Bear debate on an investment thesis, with moderator synthesis."""
    client = openai.OpenAI()
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")

    bull_system = (
        "You are a McKinsey Bull Analyst. You identify growth opportunities, competitive advantages, "
        "and reasons to invest. Be specific with data points and strategic rationale. "
        "Acknowledge risks only to show they are manageable."
    )
    bear_system = (
        "You are a McKinsey Bear Analyst. You identify risks, market headwinds, competitive threats, "
        "and reasons for caution. Be specific with data points and risk scenarios. "
        "Acknowledge strengths only to show they are insufficient."
    )
    moderator_system = (
        "You are a McKinsey Senior Partner moderating an investment debate. "
        "Synthesize the bull and bear arguments into a balanced recommendation. "
        "Provide: (1) the strongest point from each side, (2) your overall recommendation "
        "(INVEST / PASS / CONDITIONAL INVEST), and (3) key conditions or milestones to monitor."
    )

    debate_history = []

    print(f"INVESTMENT THESIS: {thesis_statement}")
    print("=" * 60)

    for round_num in range(1, num_rounds + 1):
        print(f"\n--- ROUND {round_num} ---")

        # Bull argues
        bull_messages = [
            {"role": "system", "content": bull_system},
            {"role": "user", "content": f"Investment thesis: {thesis_statement}"}
        ]
        if debate_history:
            bull_messages.append({"role": "user", "content": f"Previous debate points:\n" + "\n".join(debate_history) + "\n\nProvide your next argument."})

        bull_response = client.chat.completions.create(
            model=model, messages=bull_messages,
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
        )
        bull_arg = bull_response.choices[0].message.content
        debate_history.append(f"BULL (Round {round_num}): {bull_arg}")
        print(f"\nBULL ANALYST:\n{bull_arg}")

        # Bear responds
        bear_messages = [
            {"role": "system", "content": bear_system},
            {"role": "user", "content": f"Investment thesis: {thesis_statement}\n\nPrevious debate points:\n" + "\n".join(debate_history) + "\n\nProvide your counter-argument."}
        ]
        bear_response = client.chat.completions.create(
            model=model, messages=bear_messages,
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
        )
        bear_arg = bear_response.choices[0].message.content
        debate_history.append(f"BEAR (Round {round_num}): {bear_arg}")
        print(f"\nBEAR ANALYST:\n{bear_arg}")

    # Moderator synthesizes
    print("\n" + "=" * 60)
    print("MODERATOR SYNTHESIS")
    print("=" * 60)

    moderator_response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": moderator_system},
            {"role": "user", "content": f"Investment thesis: {thesis_statement}\n\nFull debate transcript:\n" + "\n\n".join(debate_history) + "\n\nProvide your synthesis and recommendation."}
        ],
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "500"))
    )
    synthesis = moderator_response.choices[0].message.content
    print(f"\n{synthesis}")

    return {"debate_history": debate_history, "synthesis": synthesis}

# Test
result = strategic_debate(
    "A global pharmaceutical company should acquire a mid-size biotech firm "
    "specializing in mRNA therapeutics for $2.5B, representing a 40% premium to market cap.",
    num_rounds=2
)

### Task 8: Guardrails & Output Validation for Client Deliverables

**Goal:** Build an output validation system that checks LLM-generated content against a McKinsey quality checklist and automatically retries if the output fails validation.

**Key Teaching Points:**
- Production LLM systems need guardrails — the model can produce outputs that are technically correct but fail quality standards.
- This pattern uses an LLM-as-judge to evaluate outputs against specific criteria, mirroring McKinsey's quality review process.
- Auto-retry with feedback loops allow the system to self-correct, reducing the need for manual intervention.
- Guardrails are essential for client-facing deliverables where quality and consistency are non-negotiable.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct:** Present guardrails as the 'quality review process.' Have students define a checklist, build an LLM-as-judge evaluator, then implement a retry loop. Test with deliberately flawed outputs.
>
> **Common Student Mistakes:**
> - Making the quality checklist too vague for the LLM judge
> - Not implementing a max retry limit, risking infinite loops
> - Using the same model for generation and judging without different prompts
> - Not logging why outputs fail validation
>
> **Skippable?** YES
> Output validation is important but reinforced throughout Days 2-3. If short on time, show the solution and discuss LLM-as-judge pattern.
> ---


In [ ]:
# Task 8 Solution: Guardrails & Output Validation for Client Deliverables

def validated_deliverable(topic, max_retries=3):
    """Generate a McKinsey deliverable with automatic quality validation and retry."""
    client = openai.OpenAI()
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")

    # Quality checklist for McKinsey deliverables
    quality_checklist = [
        "Contains a clear executive summary (2-3 sentences)",
        "Includes at least 3 specific, quantitative data points or estimates",
        "Provides actionable recommendations (not just observations)",
        "Uses MECE structure (mutually exclusive, collectively exhaustive)",
        "Ends with concrete next steps or a call to action"
    ]

    generation_prompt = f"""Write a McKinsey-style strategic briefing on: {topic}

Requirements:
- Start with a 2-3 sentence executive summary
- Include specific data points and market estimates
- Provide 3-4 actionable recommendations
- Use MECE structure
- End with concrete next steps

Keep the briefing concise but partner-ready (300-400 words)."""

    for attempt in range(1, max_retries + 1):
        print(f"\n{'=' * 60}")
        print(f"ATTEMPT {attempt}/{max_retries}")
        print(f"{'=' * 60}")

        # Generate the deliverable
        gen_response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a McKinsey engagement manager preparing a client deliverable. Write with precision, structure, and actionable insights."},
                {"role": "user", "content": generation_prompt}
            ],
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "600"))
        )
        deliverable = gen_response.choices[0].message.content
        print(f"\nGenerated deliverable:\n{deliverable[:200]}...")

        # Validate against quality checklist using LLM-as-judge
        checklist_str = "\n".join([f"{i+1}. {c}" for i, c in enumerate(quality_checklist)])

        val_response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a McKinsey quality reviewer. Evaluate the deliverable against each checklist item. Be strict but fair."},
                {"role": "user", "content": f"Evaluate this deliverable against the quality checklist.\n\nDELIVERABLE:\n{deliverable}\n\nCHECKLIST:\n{checklist_str}\n\nReturn JSON with keys: passed (boolean), score (number of checks passed out of {len(quality_checklist)}), failures (list of failed check descriptions), feedback (string with specific improvement suggestions)."}
            ],
            response_format={"type": "json_object"}
        )
        validation = json.loads(val_response.choices[0].message.content)

        print(f"\nValidation: {validation.get('score', 0)}/{len(quality_checklist)} checks passed")
        if validation.get("failures"):
            print(f"Failed checks: {validation['failures']}")

        if validation.get("passed", False) or validation.get("score", 0) >= len(quality_checklist):
            print("\nPASSED quality validation!")
            return {"deliverable": deliverable, "validation": validation, "attempts": attempt}

        # If not passed, add feedback to generation prompt for retry
        print(f"Retrying with feedback...")
        generation_prompt += f"\n\nPREVIOUS ATTEMPT FEEDBACK (fix these issues):\n{validation.get('feedback', 'Improve overall quality')}"

    print(f"\nMax retries ({max_retries}) reached. Returning best effort.")
    return {"deliverable": deliverable, "validation": validation, "attempts": max_retries}

# Test
result = validated_deliverable("The strategic implications of generative AI adoption for a top-10 US bank")

---
## Summary

### Key Takeaways

1. **Zero-shot vs. Few-shot**: Few-shot prompting provides examples that guide the model toward consistent format and behavior. Use it when you need predictable, structured responses.

2. **Chain-of-Thought (CoT)**: Explicitly asking for step-by-step reasoning improves accuracy on complex tasks. CoT is especially valuable for math, logic, and multi-step problems.

3. **Role-Based Prompting**: System prompts are the primary mechanism for defining agent personas. A well-crafted system prompt shapes the entire behavior profile of an agent.

4. **Structured Outputs**: JSON mode ensures machine-readable responses, which is critical for agentic workflows where outputs feed into downstream code or other agents.

5. **Multi-Turn Management**: Maintaining conversation history enables context-aware interactions, but requires careful management of the context window as conversations grow.

6. **LangChain Prompt Templates**: LangChain's `ChatPromptTemplate` and `StructuredOutputParser` provide reusable, composable abstractions for prompt engineering. The pipe operator (`|`) lets you chain prompts, LLMs, and parsers into end-to-end workflows.

### Looking Ahead

These prompt engineering techniques form the foundation for building agentic systems. In the next session, we will explore how to evaluate and measure the quality of agent outputs systematically.

In [ ]:
# Task 4 Solution: Build a Multi-Step Reasoning Agent with Tool Descriptions

tools = [
    {"name": "calculator", "description": "Performs mathematical calculations", "parameters": "expression (string)"},
    {"name": "search", "description": "Searches for factual information", "parameters": "query (string)"},
    {"name": "weather", "description": "Gets current weather for a location", "parameters": "city (string)"}
]

class ToolAgent:
    def __init__(self):
        self.client = openai.OpenAI()
        tool_descriptions = "\n".join([f"- {t['name']}: {t['description']} (params: {t['parameters']})" for t in tools])
        self.system_prompt = f"""You are an AI agent with access to the following tools:

{tool_descriptions}

When you need to use a tool, respond with JSON:
{{"thought": "your reasoning", "tool": "tool_name", "input": "tool_input"}}

When you have enough information to answer, respond with JSON:
{{"thought": "final reasoning", "tool": "final_answer", "input": "your complete answer"}}

Process one step at a time."""
    
    def simulate_tool(self, tool_name, tool_input):
        simulated = {
            "calculator": f"Result: {tool_input} = [calculated value]",
            "search": f"Search results for '{tool_input}': [relevant information found]",
            "weather": f"Weather in {tool_input}: 72\u00b0F, partly cloudy"
        }
        return simulated.get(tool_name, "Tool not found")
    
    def process(self, question):
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": question}
        ]
        
        for step in range(5):  # Max 5 steps
            response = self.client.chat.completions.create(
                model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
                messages=messages,
                response_format={"type": "json_object"},
                max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
            )
            result = json.loads(response.choices[0].message.content)
            print(f"\nStep {step + 1}:")
            print(f"  Thought: {result.get('thought', 'N/A')}")
            print(f"  Tool: {result.get('tool', 'N/A')}")
            print(f"  Input: {result.get('input', 'N/A')}")
            
            if result.get("tool") == "final_answer":
                print(f"\nFinal Answer: {result['input']}")
                return result["input"]
            
            observation = self.simulate_tool(result["tool"], result["input"])
            print(f"  Observation: {observation}")
            messages.append({"role": "assistant", "content": response.choices[0].message.content})
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        
        return "Max steps reached"

agent = ToolAgent()
agent.process("What's 15% tip on a $85 dinner, and what's the weather like in New York?")

---
## Summary

### Key Takeaways

1. **Zero-shot vs. Few-shot**: Few-shot prompting provides examples that guide the model toward consistent format and behavior. Use it when you need predictable, structured responses.

2. **Chain-of-Thought (CoT)**: Explicitly asking for step-by-step reasoning improves accuracy on complex tasks. CoT is especially valuable for math, logic, and multi-step problems.

3. **Role-Based Prompting**: System prompts are the primary mechanism for defining agent personas. A well-crafted system prompt shapes the entire behavior profile of an agent.

4. **Structured Outputs**: JSON mode ensures machine-readable responses, which is critical for agentic workflows where outputs feed into downstream code or other agents.

5. **Multi-Turn Management**: Maintaining conversation history enables context-aware interactions, but requires careful management of the context window as conversations grow.

### Looking Ahead

These prompt engineering techniques form the foundation for building agentic systems. In the next session, we will explore how to evaluate and measure the quality of agent outputs systematically.